# Email Triage Agent with Long-Term Memory (LangGraph)

An assistant sorts Gurveer's email into **ignore / notify / respond**. General rules get it most of the way, but every
person has preferences no rulebook states: *CI failures on my repos: tell me. Recruiters for sales roles: ignore.*
This notebook tests whether the agent **learns those preferences from corrections and remembers them**.

1. **Data:** 30 labelled emails (fictional senders). 8 are corrections Gurveer gives the assistant, each with a
   one-line reason; 22 are held out for testing. 11 test emails fall under a preference only the corrections state.
2. **Baseline:** triage the 22 test emails with the general rules only.
3. **Memory:** save the 8 corrections to a LangGraph store with semantic search, write it to disk, and reload it in
   a fresh store, as a new session would.
4. **With memory:** triage the same 22 emails; the 3 most similar past corrections go into the prompt.
5. **Respond:** for emails that need a reply, a tool-calling agent checks the calendar and drafts a reply (nothing is sent).

Needs: LM Studio running with the model in `LLM_MODEL` (see the repo's `.env.example`) for steps 2, 4 and 5.
The preferences in the corrections are illustrative.

In [1]:
# Cell 1 — Setup
import json, os, sys
from datetime import date
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
PROJECT_DIR = Path.cwd().parent if Path.cwd().name in ("notebooks", "src") else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR / "src"))
import triage as tr, evaluate as ev, respond
from memory import TriageMemory

OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 160)

In [2]:
# Cell 2 — The emails: 8 corrections to learn from, 22 to test on
emails = [json.loads(l) for l in (PROJECT_DIR / "eval" / "emails.jsonl").read_text(encoding="utf-8").splitlines()]
teach = [e for e in emails if e["split"] == "teach"]
test = [e for e in emails if e["split"] == "test"]
by_id = {e["id"]: e for e in emails}
display(pd.DataFrame(emails).pivot_table(index="split", columns="label", values="id", aggfunc="count"))
print("Corrections Gurveer gives the assistant:")
display(pd.DataFrame(teach)[["id", "subject", "label", "reason"]])

label,ignore,notify,respond
split,,,
teach,4,2,2
test,8,7,7


Corrections Gurveer gives the assistant:


,id,subject,label,reason
0,T1,[sidhugatwit/ds-portfolio] Run failed: notebooks - main (3f2a91c),notify,"CI failures on my repos: tell me, but there's nobody to reply to."
1,T2,"Machine Learning Intern openings, Summer 2027",respond,"Recruiters for AI, ML or data roles: always respond, even to a template."
2,T3,Quick call this week?,ignore,"Recruiters for sales or roles unrelated to data and AI: ignore, even when they ask a question."
3,T4,Spring Involvement Fair this Thursday!,ignore,Campus-wide event and club announcements: ignore.
4,T5,Seminar: Night Operations Safety (WINGS credit),notify,FAA Safety Team notices: tell me; they count toward WINGS and keep me current.
5,T6,Your receipt from Lovable #2041-7730,ignore,"Receipts and renewals: ignore, I check billing myself."
6,T7,URGENT: Your account will be suspended,ignore,Phishing.
7,T8,Could you share your clustering notebook?,respond,A direct request from a classmate.


In [3]:
# Cell 3 — Embeddings for memory search: BGE-small (downloads ~130 MB the first time)
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-small-en-v1.5")
DIMS = model.get_sentence_embedding_dimension()
def embed(texts):
    return model.encode(list(texts), normalize_embeddings=True).tolist()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

C:\Users\sidhu\AppData\Local\Temp\ipykernel_21244\1094100336.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  DIMS = model.get_sentence_embedding_dimension()


In [4]:
# Cell 4 — Baseline: general rules only, no memory
chat = tr.openai_chat()
baseline = ev.run_triage(tr.build_graph(chat), test)
display(ev.accuracy_table({"no memory": baseline}))
print("Mistakes:")
display(baseline.loc[~baseline.correct, ["id", "group", "label", "predicted", "reasoning"]])

,no memory
all,17/22
general,11/11
preference,6/11
unparsed,0


Mistakes:


,id,group,label,predicted,reasoning
0,X1,preference,notify,ignore,This is an automated system alert from GitHub regarding a failed workflow run on a personal repository. It falls under the automated notifications category ...
1,X2,preference,notify,ignore,This email is an automated system notification from GitHub reporting a failed CI/CD build for a repository. It falls under the 'automated notifications' cat...
4,X5,preference,ignore,respond,The email contains a direct request from a recruiter asking if Gurveer is interested in a position and proposing a phone call. It requires a personal reply ...
7,X8,preference,notify,ignore,"This is a mass promotional announcement for a webinar sent from a noreply address, which aligns with the ignore category for automated notifications and mar..."
9,X10,preference,ignore,notify,This is a personal order shipping update that provides important tracking information but requires no action or reply.


In [5]:
# Cell 5 — Teach: save the 8 corrections to long-term memory, and write the store to disk
memory = TriageMemory(embed, DIMS)
for e in teach:
    memory.add(e["id"], e, e["label"], e["reason"])
memory.save(OUTPUT_DIR / "memory_store.json")
print(f"Saved {len(memory)} corrections to outputs/memory_store.json")

Saved 8 corrections to outputs/memory_store.json


In [6]:
# Cell 6 — New session: reload memory from disk into a fresh store, and check what it recalls for each test email
memory = TriageMemory.load(OUTPUT_DIR / "memory_store.json", embed, DIMS)
recall = pd.DataFrame([{"id": e["id"], "group": e["group"], "subject": e["subject"][:60],
                        "recalled (most similar first)": [m["key"] for m in memory.search(e, 3)]} for e in test])
display(recall)

,id,group,subject,recalled (most similar first)
0,X1,preference,[sidhugatwit/ds-portfolio] Run failed: weekly-heavy - main (,"[T1, T8, T3]"
1,X2,preference,[sidhugatwit/runway-to-results] Run failed: build - rebuild-,"[T1, T8, T3]"
2,X3,preference,Data Science Co-op (Fall 2027): applications open,"[T2, T8, T3]"
3,X4,preference,Join our AI Engineering intern talent community,"[T2, T8, T3]"
4,X5,preference,Store Manager opportunity: interested?,"[T3, T8, T4]"
5,X6,preference,Homecoming Week schedule is here!,"[T4, T8, T5]"
6,X7,preference,Robotics Club open house: all majors welcome,"[T4, T8, T5]"
7,X8,preference,Webinar: Runway Incursion Avoidance,"[T5, T3, T2]"
8,X9,preference,Safety notice: winter flying and carburetor icing,"[T5, T3, T2]"
9,X10,preference,Your order #88213 has shipped,"[T6, T3, T7]"


In [7]:
# Cell 7 — With memory: the same 22 emails, now with the 3 most similar past corrections in the prompt
with_memory = ev.run_triage(tr.build_graph(chat, memory), test)
display(ev.accuracy_table({"no memory": baseline, "with memory": with_memory}))
changes = baseline[["id", "group", "label"]].assign(before=baseline.predicted, after=with_memory.predicted,
                                                    subject=[by_id[i]["subject"][:60] for i in baseline.id])
print("Emails whose triage changed:")
display(changes[changes.before != changes.after])
print("Still wrong with memory:")
display(with_memory.loc[~with_memory.correct, ["id", "group", "label", "predicted", "memories_used", "reasoning"]])

,no memory,with memory
all,17/22,22/22
general,11/11,11/11
preference,6/11,11/11
unparsed,0,0


Emails whose triage changed:


,id,group,label,before,after,subject
0,X1,preference,notify,ignore,notify,[sidhugatwit/ds-portfolio] Run failed: weekly-heavy - main (
1,X2,preference,notify,ignore,notify,[sidhugatwit/runway-to-results] Run failed: build - rebuild-
4,X5,preference,ignore,respond,ignore,Store Manager opportunity: interested?
7,X8,preference,notify,ignore,notify,Webinar: Runway Incursion Avoidance
9,X10,preference,ignore,notify,ignore,Your order #88213 has shipped


Still wrong with memory:


,id,group,label,predicted,memories_used,reasoning


In [8]:
# Cell 8 — Respond: for emails that need a reply, a tool-calling agent checks the calendar and drafts one (not sent)
responder = respond.make_responder(respond.lmstudio_llm())
full_graph = tr.build_graph(chat, memory, responder)
replies = {}
for email_id in ["X16", "X17", "X22"]:
    out = full_graph.invoke({"email": by_id[email_id]})
    replies[email_id] = {"classification": out.get("classification"), **out.get("reply", {})}
    print(f"=== {email_id}: {by_id[email_id]['subject']} -> {out.get('classification')}")
    for call in replies[email_id].get("tool_calls", []):
        print("  tool:", json.dumps(call, ensure_ascii=False)[:400])
    print()

=== X16: Move Saturday's lesson to 10? -> respond
  tool: {"tool": "check_calendar_availability", "day": "Saturday", "result": []}
  tool: {"tool": "write_email", "to": "sam.ortiz@bluehill-aviation.example", "subject": "Re: Move Saturday's lesson to 10?", "content": "Hi Sam,\n\nThanks for the update. Unfortunately, I don't have any availability on Saturday, so I won't be able to move the lesson to 10:00.\n\nLet me know if you'd like to reschedule for a different day.\n\nBest,\nGurveer"}

=== X17: Co-op planning meeting -> respond
  tool: {"tool": "check_calendar_availability", "day": "monday", "result": ["10:00 AM", "2:00 PM"]}
  tool: {"tool": "check_calendar_availability", "day": "tuesday", "result": ["9:00 AM", "1:00 PM"]}
  tool: {"tool": "check_calendar_availability", "day": "wednesday", "result": ["11:00 AM", "3:00 PM"]}
  tool: {"tool": "check_calendar_availability", "day": "thursday", "result": ["10:00 AM", "4:00 PM"]}
  tool: {"tool": "check_calendar_availability", "day": "fri

In [9]:
# Cell 9 — Save the run
record = {"date": date.today().isoformat(), "model": os.getenv("LLM_MODEL"), "k": 3,
          "baseline": baseline.to_dict(orient="records"), "with_memory": with_memory.to_dict(orient="records"),
          "replies": replies}
(OUTPUT_DIR / "triage_run.json").write_text(json.dumps(record, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("Saved outputs/triage_run.json")

Saved outputs/triage_run.json
